# Lesson 4.1 — Rank, Range, and Null Space
**Module 6 · Unit 4 · Lesson 13**

Capability first: the **range** of J = available tool directions, its complement = impossible directions, and the **null space** = internal self-motion. We exhibit a redundant arm's null-space motion and a straight arm's impossible direction.

In [1]:
import numpy as np
def dh(th,d,a,al):
    ct,st,ca,sa=np.cos(th),np.sin(th),np.cos(al),np.sin(al)
    return np.array([[ct,-st*ca,st*sa,a*ct],[st,ct*ca,-ct*sa,a*st],[0,sa,ca,d],[0,0,0,1]])
def forward_chain(P,T,q):
    M=np.eye(4); Ms=[M.copy()]
    for i,(th0,d0,a,al) in enumerate(P):
        th,d=(th0+q[i],d0) if T[i]=="R" else (th0,d0+q[i]); M=M@dh(th,d,a,al); Ms.append(M.copy())
    return Ms
def geometric_jacobian(P,T,q):
    Ms=forward_chain(P,T,q); on=Ms[-1][:3,3]; J=np.zeros((6,len(q)))
    for i in range(len(q)):
        z=Ms[i][:3,2]; o=Ms[i][:3,3]
        if T[i]=="R": J[:3,i]=np.cross(z,on-o); J[3:,i]=z
        else: J[:3,i]=z
    return J
def Jv_planar(P,T,q): return geometric_jacobian(P,T,q)[:2,:]  # x,y rows for planar arms


## Redundant planar 3R arm: a 1-D null space (internal self-motion)

In [2]:
checks=[]
P3=[(0,0,1,0),(0,0,1,0),(0,0,0.6,0)]; T3=["R","R","R"]; q=np.array([0.3,0.5,-0.4])
Jv=Jv_planar(P3,T3,q)              # 2x3
U,S,Vt=np.linalg.svd(Jv)
print("singular values:",np.round(S,3)," rank:",np.linalg.matrix_rank(Jv)," null-space dim:",3-np.linalg.matrix_rank(Jv))
q_null=Vt[-1]                      # right-singular vector with ~0 singular value
print("null-space joint velocity:",np.round(q_null,3)," -> tool velocity:",np.round(Jv@q_null,8))
checks.append(np.allclose(Jv@q_null,0,atol=1e-9))

singular values: [3.033 0.202]  rank: 2  null-space dim: 1
null-space joint velocity: [-0.381  0.283  0.88 ]  -> tool velocity: [-0. -0.]


## Straight planar 2R arm: rank drops, a direction becomes impossible

In [3]:
P2=[(0,0,1,0),(0,0,1,0)]; T2=["R","R"]
Jstr=Jv_planar(P2,T2,np.array([0.4,0.0]))   # theta2=0 -> straight
Us,Ss,Vts=np.linalg.svd(Jstr)
print("straight-pose singular values:",np.round(Ss,4)," rank:",np.linalg.matrix_rank(Jstr))
print("impossible direction (orthogonal complement of range):",np.round(Us[:,-1],3))
checks.append(np.linalg.matrix_rank(Jstr)<2 and Ss[-1]<1e-6)

straight-pose singular values: [2.2361 0.    ]  rank: 1
impossible direction (orthogonal complement of range): [0.921 0.389]


## Summary of capability buckets

In [4]:
print("range      -> available tool directions")
print("complement -> impossible directions (appear when rank drops)")
print("null space -> internal self-motion (tool unaffected)")
assert all(checks), f"FAILED: {checks}"
print("All checks passed.")

range      -> available tool directions
complement -> impossible directions (appear when rank drops)
null space -> internal self-motion (tool unaffected)
All checks passed.
